# Health Insurance Pricing Case: Clustering and Linear Regression Comparison

## Business Problem

A health insurance company needs better analytical support to define how much to charge for its health plans.

The pricing decision must balance two business goals:

- charge enough to reduce the risk of financial losses;
- remain attractive enough to support customer acquisition.

This project investigates whether different analytical approaches can help explain customer risk and support more informed premium-setting decisions.


## Project Objective

This project compares two approaches for solving the same health insurance pricing case:

1. a clustering-based approach, rebuilt from the completed health insurance case;
2. a linear regression approach, using multiple regression and model refinement through stepwise AIC evaluation.

The goal is to evaluate how each method performs on unseen data and how useful each one is for pricing decisions.


## Analytical Strategy

To ensure a fair comparison, both approaches will be evaluated under the same logic:

1. split the historical dataset into 80% training data and 20% test data;
2. rebuild the clustering workflow using the training set;
3. evaluate clustering on the holdout set and on 3 new incoming clients;
4. rebuild the case with linear regression using the same train-test split logic;
5. compare the predictive behavior, interpretability, and business usefulness of both approaches.


## Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import joblib


In [2]:
# =========================================
# Notebook configuration
# =========================================

RANDOM_STATE = 1

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

## Data Loading

In [3]:
# =========================================
# Reading csv file
# =========================================
base_health_insurance_df = pd.read_csv("../data/base_health_insurance.csv",sep = ";")
display(base_health_insurance_df.head()) 

,age,gender,bmi,children,discount_eligibility,region,expenses,premium
0,19,female,"24,1",0,yes,south,"1483,21","1744,95"
1,18,male,"23,5",1,no,southeast,"2237,61","2632,48"
2,28,male,33,3,no,southeast,"2046,09","2728,12"
3,33,male,"23,3",0,no,midwest,"2331,69","2743,17"
4,32,male,"23,4",0,no,midwest,"1942,5","2285,3"


## Data Validation

In [4]:
# =========================================
# checking data types and missing values
# =========================================

print(base_health_insurance_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   age                   1338 non-null   int64
 1   gender                1338 non-null   str  
 2   bmi                   1338 non-null   str  
 3   children              1338 non-null   int64
 4   discount_eligibility  1338 non-null   str  
 5   region                1338 non-null   str  
 6   expenses              1338 non-null   str  
 7   premium               1338 non-null   str  
dtypes: int64(2), str(6)
memory usage: 83.8 KB
None


## Data Adjustment

Before proceeding with analysis, certain variables require type correction to ensure they are in the appropriate format for computation.

During data validation, it was identified that `bmi`, `expenses`, and `premium` were loaded as strings. Comma separators are replaced and the columns are converted to their correct numeric types. The `discount_eligibility` column is also encoded as a binary variable to enable its use in clustering.

In [5]:
# =========================================
# Data Adjustment
# =========================================

base_health_insurance_df["bmi"] = base_health_insurance_df["bmi"].str.replace(",", ".").astype(float) # replacing comma separator and converting bmi to float

base_health_insurance_df["expenses"] = base_health_insurance_df["expenses"].str.replace(",", ".").astype(float) # replacing comma separator and converting expenses to float

base_health_insurance_df["premium"] = base_health_insurance_df["premium"].str.replace(",", ".").astype(float) # replacing comma separator and converting premium to float

base_health_insurance_df["discount_eligibility_binary"] = np.where(
    base_health_insurance_df["discount_eligibility"] == "yes", 1, 0) # encoding discount eligibility as binary: 1 = yes, 0 = no

display(base_health_insurance_df["discount_eligibility"].unique()) # verifying unique values of original column

print(base_health_insurance_df.info()) # confirming updated data types


<StringArray>
['yes', 'no']
Length: 2, dtype: str

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   age                          1338 non-null   int64  
 1   gender                       1338 non-null   str    
 2   bmi                          1338 non-null   float64
 3   children                     1338 non-null   int64  
 4   discount_eligibility         1338 non-null   str    
 5   region                       1338 non-null   str    
 6   expenses                     1338 non-null   float64
 7   premium                      1338 non-null   float64
 8   discount_eligibility_binary  1338 non-null   int64  
dtypes: float64(3), int64(3), str(3)
memory usage: 94.2 KB
None


## Train-Test Split (80/20)

To evaluate model performance on unseen data, the dataset is split into two subsets:

- **Training set (80%)**: used to fit the models and learn patterns from historical data.
- **Test set (20%)**: kept separate and used only for out-of-sample evaluation.

This separation helps reduce optimistic bias and provides a more realistic estimate of how the approach may perform in real pricing decisions.


In [6]:
from sklearn.model_selection import train_test_split  # function to split data into train and test sets

# splitting the full dataset: 80% for training and 20% for testing
base_health_insurance_train_df, base_health_insurance_test_df = train_test_split(
    base_health_insurance_df,  # source dataframe to split
    train_size=0.80,  # proportion for training set
    random_state=RANDOM_STATE,  # ensures reproducible split
    shuffle=True  # randomly shuffles rows before splitting
)

# resetting index in the training dataframe to keep a clean sequential index
base_health_insurance_train_df = base_health_insurance_train_df.reset_index(drop=True)  # drop old index

# resetting index in the test dataframe to keep a clean sequential index
base_health_insurance_test_df = base_health_insurance_test_df.reset_index(drop=True)  # drop old index

print("Train shape:", base_health_insurance_train_df.shape)  # shows training rows and columns
print("Test shape:", base_health_insurance_test_df.shape)  # shows test rows and columns



Train shape: (1070, 9)
Test shape: (268, 9)
